In [1]:
# Load the autoreload extension
%load_ext autoreload

# Set autoreload mode
%autoreload 2

# Label Mapping

In [2]:
import requests
import pandas as pd

The Label Mapping Endpoint recently had another 125 new maps added  

In [ ]:
mapping_endpoint ="https://api.datamermaid.org/v1/classification/labelmappings/?provider=CoralNet" 

response = requests.get(mapping_endpoint)
data = response.json()
labelset = data["results"]

while data["next"]:
    response = requests.get(data["next"])
    data = response.json()
    labelset.extend(data["results"])
label_mapping = {
    label["provider_id"]: label["benthic_attribute_name"] for label in labelset
}

In [ ]:
df_mapping_orig = pd.read_csv("dataframes/CoralNetMermaidMatchedCoralFocusModel2Reassign_20240503.csv")
df_mapping_upd = pd.read_csv("dataframes/class_mapping_concepts_2026_01_16.csv")

## Original Mapping
https://docs.google.com/spreadsheets/d/1BUk4hK78CxwbFWLSeu6taFDJws1z6e96WTNW6E8vmWo/edit?gid=1827266844#gid=1827266844

The original label mapping dictionary consists of 711 (710 in the endpoint) CoralNet labels (IDs) that have been mapped to 305 unique benthic attributes, 12 unique growth forms for a total of 344 unique benthic attribute - growth form combinations.

When using the focus labels (CoralFocus3Label), there are 55 unique BA+GF combinations (+1 remove label), with 119 labels being removed from the final mapping.

In [6]:
df_mapping_orig.head(3)

,CoralNetID,CoralNetName,Default short code,Functional group,Public sources using,Public annotation count,NewCoralNetID,Top100,MERMAID_BA,BA_AssignNotes,...,CoralFocusLabel,precision_CoralFocusModel,recall_CoralFocusModel,f1-score_CoralFocusModel,support_CoralFocusModel,CoralFocus2Label,CoralFocus3Label,CoralFocus3_BaID,CoralFocus3_GfID,Unnamed: 44
0,82,Turf algae,Turf,Algae,321,5496896,82,1,Turf algae,NaN,...,Turf algae,0.605677,0.356949,0.449179,9923,Turf algae,Turf algae,20090bf4-868e-431b-974c-ab9be5bbdb5f,NaN,NaN
1,84,Sand,Sand,Soft Substrate,418,1882701,84,1,Sand,NaN,...,Sand,0.749551,0.706228,0.727245,10052,Sand,Sand,b76bca12-884b-4404-bb9f-97d505b0fe58,NaN,NaN
2,1348,CRED-Turf growing on hard substrate,*TURFH,Hard Substrate,18,1216402,82,1,Turf algae,NaN,...,Turf algae,0.605677,0.356949,0.449179,9923,Turf algae,Turf algae,20090bf4-868e-431b-974c-ab9be5bbdb5f,NaN,NaN


In [7]:
len(label_mapping), df_mapping_orig.shape

(836, (711, 45))

In [8]:
df_mapping_orig["MERMAID_BA"].nunique(), df_mapping_orig["MERMAID_GF"].nunique(), df_mapping_orig["ba_gf"].nunique()

(305, 12, 344)

In [9]:
print("Number of labels who have been removed from the final mapping:", df_mapping_orig[df_mapping_orig["CoralFocus3Label"]=="Remove"].shape[0])

Number of labels who have been removed from the final mapping: 119


In [10]:
df_mapping_orig["CoralFocusLabel"].nunique(), df_mapping_orig["CoralFocus2Label"].nunique(), df_mapping_orig["CoralFocus3Label"].nunique()

(70, 63, 56)

## New Label Mapping (Jonathan's)
https://docs.google.com/spreadsheets/d/1DjE7SpXE2AaKsJXbqNm18ClGIGp5nAmwO9LhmG9yzTQ/edit?usp=sharing

The updated label mapping dictionary consists of 1320 unique labels from three sources (778 from MERMAID, 711 from CoralNet, and 95 from Coralscapes, resulting in a total of 1584 labels) that have been mapped to 896 unique labels. 

In [11]:
df_mapping_upd.head(3)

,name,source,should_map_to_label,should_map_to_label_source,kingdom,phylum,order,family,genus,species,...,foliose,oval,dead,bleached,algae,background,anthropogenic,trash,transect,class
0,acropora alive,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,False,False,False,False,False,False,False,False,hexacorallia
1,acropora bleached,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,False,False,True,False,False,False,False,False,hexacorallia
2,acropora dead,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,False,True,False,False,False,False,False,False,hexacorallia


In [12]:
df_mapping_upd.shape, df_mapping_upd["name"].nunique()

((1584, 44), 1320)

In [13]:
df_mapping_upd["source"].value_counts()

source
mermaid        778
coralnet       711
coralscapes     95
Name: count, dtype: int64

In [14]:
tmp = df_mapping_upd["name"].copy()
tmp = tmp.mask(df_mapping_upd["should_map_to_label"].notna(), df_mapping_upd["should_map_to_label"])
print("The updated number of unique labels in the mapping is", tmp.nunique())

The updated number of unique labels in the mapping is 896


## Differences between original and updated coral mapping

In [15]:
tmp = df_mapping_upd[df_mapping_upd["source"]=="coralnet"][["name", "should_map_to_label", "should_map_to_label_source"]].copy()
tmp2 = df_mapping_orig[["CoralNetName", "ba_gf", "CoralFocus3Label"]]
tmp2 = tmp2.apply(lambda s: s.str.lower() if s.dtype == "object" else s)
df_mapping_comparison = tmp.merge(tmp2, left_on = "name", right_on = "CoralNetName", how = "outer")

In [16]:
df_mapping_comparison.head(3)

,name,should_map_to_label,should_map_to_label_source,CoralNetName,ba_gf,CoralFocus3Label
0,acanthastrea,acanthastrea,mermaid,acanthastrea,acanthastrea,remove
1,acanthophora spicifera,acanthophora spicifera,mermaid,acanthophora spicifera,acanthophora spicifera,macroalgae
2,acetabularia spp.,acetabularia,mermaid,acetabularia spp.,acetabularia,macroalgae


In [17]:
print("Number of labels that have the same mapping in the original and updated mapping:",)
print((df_mapping_comparison["should_map_to_label"]== df_mapping_comparison["ba_gf"]).value_counts())

Number of labels that have the same mapping in the original and updated mapping:
True     406
False    313
Name: count, dtype: int64


In [18]:
df_mapping_comparison[(df_mapping_comparison["should_map_to_label"]!=df_mapping_comparison["ba_gf"])]

,name,should_map_to_label,should_map_to_label_source,CoralNetName,ba_gf,CoralFocus3Label
3,acropora,acropora alive,coralscapes,acropora,acropora,acropora
4,acropora (arborescent table),table acropora alive,coralscapes,acropora (arborescent table),acropora arborescent,remove
5,acropora (arborescent),acropora alive,coralscapes,acropora (arborescent),acropora arborescent,remove
6,acropora (branching),acropora alive,coralscapes,acropora (branching),acropora branching,acropora - branching
7,acropora (corymbose),acropora alive,coralscapes,acropora (corymbose),acropora corymbose,remove
...,...,...,...,...,...,...
708,wand,transect tools,coralscapes,wand,other,remove
709,water,background,coralscapes,water,other,remove
710,worms: polychaetes,polychaeta,coralnet,worms: polychaetes,other invertebrates,other invertebrates
711,worms: polychaetes: tube worms,polychaeta,coralnet,worms: polychaetes: tube worms,other invertebrates,other invertebrates


In [19]:
print("Number of labels that have the same mapping in the original and updated mapping (using the CoralFocus3Label column):",)
print((df_mapping_comparison["should_map_to_label"]== df_mapping_comparison["CoralFocus3Label"]).value_counts())

Number of labels that have the same mapping in the original and updated mapping (using the CoralFocus3Label column):
False    611
True     108
Name: count, dtype: int64


In [20]:
print("There are 22 labels that were previously mapped to other that now have an associated class")
df_mapping_comparison[(df_mapping_comparison["ba_gf"]=="other") & 
                      (df_mapping_comparison["should_map_to_label"].notna())& 
                      (df_mapping_comparison["should_map_to_label"]!="unknown")].shape

There are 22 labels that were previously mapped to other that now have an associated class


(22, 6)

# Concept EDA

In [21]:
df_mapping_orig.head(3)

,CoralNetID,CoralNetName,Default short code,Functional group,Public sources using,Public annotation count,NewCoralNetID,Top100,MERMAID_BA,BA_AssignNotes,...,CoralFocusLabel,precision_CoralFocusModel,recall_CoralFocusModel,f1-score_CoralFocusModel,support_CoralFocusModel,CoralFocus2Label,CoralFocus3Label,CoralFocus3_BaID,CoralFocus3_GfID,Unnamed: 44
0,82,Turf algae,Turf,Algae,321,5496896,82,1,Turf algae,NaN,...,Turf algae,0.605677,0.356949,0.449179,9923,Turf algae,Turf algae,20090bf4-868e-431b-974c-ab9be5bbdb5f,NaN,NaN
1,84,Sand,Sand,Soft Substrate,418,1882701,84,1,Sand,NaN,...,Sand,0.749551,0.706228,0.727245,10052,Sand,Sand,b76bca12-884b-4404-bb9f-97d505b0fe58,NaN,NaN
2,1348,CRED-Turf growing on hard substrate,*TURFH,Hard Substrate,18,1216402,82,1,Turf algae,NaN,...,Turf algae,0.605677,0.356949,0.449179,9923,Turf algae,Turf algae,20090bf4-868e-431b-974c-ab9be5bbdb5f,NaN,NaN


In [22]:
df_mapping_upd.head(3)

,name,source,should_map_to_label,should_map_to_label_source,kingdom,phylum,order,family,genus,species,...,foliose,oval,dead,bleached,algae,background,anthropogenic,trash,transect,class
0,acropora alive,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,False,False,False,False,False,False,False,False,hexacorallia
1,acropora bleached,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,False,False,True,False,False,False,False,False,hexacorallia
2,acropora dead,coralscapes,NaN,NaN,animalia,cnidaria,scleractinia,acroporidae,acropora,not_given,...,False,False,True,False,False,False,False,False,False,hexacorallia


In [24]:
coralnet_labels_source_df = pd.read_csv("dataframes/coralnet_per_source_labels_annotations.csv").drop("Unnamed: 0", axis=1)
coralnet_labels_source_df = (
    coralnet_labels_source_df
    .groupby("Label ID", as_index=True)
    .agg(
        **{
            "Num Annotations": ("Num Annotations", "sum"),
            "Num Images": ("Num Images", "sum"),
            "Num Sources": ("Source ID", "nunique"),
        }
    )
)

In [25]:
coralnet_id2_name = df_mapping_orig[["CoralNetID", "CoralNetName"]].drop_duplicates().set_index("CoralNetID")["CoralNetName"].to_dict()
coralnet_labels_source_df = coralnet_labels_source_df.reset_index()
coralnet_labels_source_df["CoralNetName"] = coralnet_labels_source_df["Label ID"].map(coralnet_id2_name)
coralnet_labels_source_df = coralnet_labels_source_df[coralnet_labels_source_df["CoralNetName"].notna()]
coralnet_labels_source_df["name"] = coralnet_labels_source_df["CoralNetName"].str.lower()

In [26]:
coralnet_labels_source_df = coralnet_labels_source_df.merge(df_mapping_upd[df_mapping_upd["source"]=="coralnet"], on = "name", how = "left")

In [27]:
coralnet_labels_source_df.head(5)

,Label ID,Num Annotations,Num Images,Num Sources,CoralNetName,name,source,should_map_to_label,should_map_to_label_source,kingdom,...,foliose,oval,dead,bleached,algae,background,anthropogenic,trash,transect,class
0,58,2521,1763,77,Acanthastrea,acanthastrea,coralnet,acanthastrea,mermaid,animalia,...,False,False,False,False,False,False,False,False,False,hexacorallia
1,59,418545,72003,148,Acropora,acropora,coralnet,acropora alive,coralscapes,animalia,...,False,False,False,False,False,False,False,False,False,hexacorallia
2,60,18932,9101,111,Astreopora,astreopora,coralnet,astreopora,mermaid,animalia,...,False,False,False,False,False,False,False,False,False,hexacorallia
3,61,9115,5054,99,Cyphastrea,cyphastrea,coralnet,cyphastrea,mermaid,animalia,...,False,False,False,False,False,False,False,False,False,hexacorallia
4,62,8614,4900,38,Favia (Indo-Pacific),favia (indo-pacific),coralnet,favia,mermaid,animalia,...,False,False,False,False,False,False,False,False,False,hexacorallia


In [28]:
coralnet_labels_source_df.columns

Index(['Label ID', 'Num Annotations', 'Num Images', 'Num Sources',
       'CoralNetName', 'name', 'source', 'should_map_to_label',
       'should_map_to_label_source', 'kingdom', 'phylum', 'order', 'family',
       'genus', 'species', 'massive', 'brain', 'submassive', 'corymbose',
       'round', 'tubular', 'free_living', 'tabular', 'external_polyps',
       'plating', 'lobed brain', 'cup_coral', 'fleshy', 'columnar', 'solitary',
       'prostrate', 'branching', 'meandroid', 'encrusting', 'arborescent',
       'bushy', 'digitate', 'phaceloid', 'colonial', 'foliose', 'oval', 'dead',
       'bleached', 'algae', 'background', 'anthropogenic', 'trash', 'transect',
       'class'],
      dtype='object')

In [29]:
for taxonomic_rank in ["kingdom", "phylum", "class", "order", "family", "genus"]:
    taxonomic_counts = (
        coralnet_labels_source_df.assign(**{taxonomic_rank: coralnet_labels_source_df[taxonomic_rank].fillna("unknown")})
        .groupby(taxonomic_rank, as_index=False)
        .agg(
            num_annotations=("Num Annotations", "sum"),
            num_images=("Num Images", "sum"),
        )
        .sort_values("num_annotations", ascending=False)
    )
    display(taxonomic_counts.head(5))

,kingdom,num_annotations,num_images
4,none,18320375,1874425
0,animalia,7143524,1554262
5,not_given,2829178,676312
6,plantae,1210150,163909
1,bacillati,324223,84221


,phylum,num_annotations,num_images
10,none,18320375,1874425
6,cnidaria,5485773,1367720
16,rhodophyta,2249983,537704
9,mollusca,852580,30913
11,not_given,680663,47308


,class,num_annotations,num_images
15,none,18320375,1874425
11,hexacorallia,4494156,1154852
9,florideophyceae,2249983,537704
16,not_given,1746778,282444
2,bivalvia,845343,28018


,order,num_annotations,num_images
32,none,18320375,1874425
40,scleractinia,4026044,1090713
33,not_given,2733301,354065
14,corallinales,1926185,416941
2,alismatales,667243,40226


,family,num_annotations,num_images
57,none,18320375,1874425
58,not_given,5783924,926778
69,poritidae,1210890,288370
1,acroporidae,1141836,275971
29,dictyotaceae,545069,132807


,genus,num_annotations,num_images
109,none,18320375,1874425
110,not_given,5964094,967788
134,porites,1199552,284589
4,acropora,541128,102281
100,montipora,506017,144198


In [30]:
for growth_form in ['massive', 'brain', 'submassive', 'corymbose',
       'round', 'tubular', 'free_living', 'tabular', 'external_polyps',
       'plating', 'lobed brain', 'cup_coral', 'fleshy', 'columnar', 'solitary',
       'prostrate', 'branching', 'meandroid', 'encrusting', 'arborescent',
       'bushy', 'digitate', 'phaceloid', 'colonial', 'foliose']:
    growth_form_counts = (
        coralnet_labels_source_df.assign(**{growth_form: coralnet_labels_source_df[growth_form].fillna("unknown")})
        .groupby(growth_form, as_index=False)
        .agg(
            num_annotations=("Num Annotations", "sum"),
            num_images=("Num Images", "sum"),
        )
        # .sort_values("num_annotations", ascending=False)
        )
    print(growth_form)
    display(growth_form_counts.head(5))

massive


,massive,num_annotations,num_images
0,False,29123265,4143720
1,True,714188,212202
2,unknown,52141,13581


brain


,brain,num_annotations,num_images
0,False,29808456,4344456
1,True,28997,11466
2,unknown,52141,13581


submassive


...

,phaceloid,num_annotations,num_images
0,False,29833537,4354164
1,True,3916,1758
2,unknown,52141,13581


colonial


,colonial,num_annotations,num_images
0,False,29835751,4355222
1,True,1702,700
2,unknown,52141,13581


foliose


,foliose,num_annotations,num_images
0,False,29762964,4330073
1,True,74489,25849
2,unknown,52141,13581


In [31]:
for concept in ['algae', 'background', 'anthropogenic', 'trash', 'transect', 'class']:
    concept_counts = (
        coralnet_labels_source_df.assign(**{concept: coralnet_labels_source_df[concept].fillna("unknown")})
        .groupby(concept, as_index=False)
        .agg(
            num_annotations=("Num Annotations", "sum"),
            num_images=("Num Images", "sum"),
        )
        # .sort_values("num_annotations", ascending=False)
        )
    print(concept)
    display(concept_counts.head(5))

algae


,algae,num_annotations,num_images
0,False,16701643,2685884
1,True,13135810,1670038
2,unknown,52141,13581


background


,background,num_annotations,num_images
0,False,29667917,4331634
1,True,169536,24288
2,unknown,52141,13581


anthropogenic


...

,trash,num_annotations,num_images
0,False,29836867,4355603
1,True,586,319
2,unknown,52141,13581


transect


,transect,num_annotations,num_images
0,False,29708182,4307658
1,True,129271,48264
2,unknown,52141,13581


class


,class,num_annotations,num_images
0,ascidiacea,10076,2622
1,asteroidea,1637,1311
2,bivalvia,845343,28018
3,chrysophyceae,13,13
4,crinoidea,3,2


In [32]:
for concept in ['dead', 'bleached']:
    concept_counts = (
        coralnet_labels_source_df.assign(**{concept: coralnet_labels_source_df[concept].fillna("unknown")})
        .groupby(concept, as_index=False)
        .agg(
            num_annotations=("Num Annotations", "sum"),
            num_images=("Num Images", "sum"),
        )
        # .sort_values("num_annotations", ascending=False)
        )
    print(concept)
    display(concept_counts.head(5))

dead


,dead,num_annotations,num_images
0,False,29220130,4285018
1,True,617323,70904
2,unknown,52141,13581


bleached


,bleached,num_annotations,num_images
0,False,29767429,4339074
1,True,70024,16848
2,unknown,52141,13581
